# 01 — Data Quality and Source Audit

## Purpose

This notebook independently audits every raw source for missing values, duplicates, metadata/footer contamination, year and seasonal coverage, region-name consistency, units, workbook formatting, and GASTAT/CSV overlap.

No value is corrected or merged here. The notebook reports evidence and leaves transformation to Notebook 02.

## Outputs

- `results/quality_report.md`
- `results/conflict_report.csv`
- `figures/missing_values.png`
- `figures/region_name_mapping.png`
- `figures/year_coverage.png`


## 1. Imports, Paths, and Extraction Definitions


In [ ]:
from datetime import datetime
from pathlib import Path
import hashlib
import re

import numpy as np
import openpyxl
import pandas as pd

RUN_STARTED = datetime.now().astimezone()
NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name.lower() != "notebooks":
    raise RuntimeError(
        "Run this notebook from Saudi-Energy-Forecasting/notebooks/. "
        f"Current directory: {NOTEBOOK_DIR}"
    )

PROJECT_ROOT = NOTEBOOK_DIR.parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
GASTAT_DIR = RAW_DIR / "old_thesis_data"
SAUDI_ENERGY_DIR = RAW_DIR / "saudi_energy"
REFERENCE_DIR = RAW_DIR / "technical_work_reference"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

for directory in (INTERIM_DIR, PROCESSED_DIR, RESULTS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

HOUSEHOLD_CSV_NAME = (
    "houses-consumption-and-cost-of-electricity-in-the-administrative-regions- (2).csv"
)
HOUSEHOLD_CSV_PATH = SAUDI_ENERGY_DIR / HOUSEHOLD_CSV_NAME

GASTAT_TABLES = {
    2019: {
        "file": "HouseholdEnergyStatistics2019En.xlsx",
        "sheet": "76",
        "table": "Table 37",
        "data_rows": (8, 20),
    },
    2020: {
        "file": "HouseholdEnergyStatistics2020En.xlsx",
        "sheet": "25",
        "table": "Houses Consumption and Cost",
        "data_rows": (8, 20),
    },
    2021: {
        "file": "HouseholdEnergyStatistics2021En.xlsx",
        "sheet": "27",
        "table": "Houses Consumption and Cost",
        "data_rows": (8, 20),
    },
    2022: {
        "file": "HouseholdEnergyStatistics2022En.xlsx",
        "sheet": "1-2",
        "table": "Table 1-2",
        "data_rows": (8, 20),
    },
}

CANONICAL_REGIONS = [
    "Riyadh",
    "Makkah",
    "Madinah",
    "Al-Qassim",
    "Eastern Region",
    "Asir",
    "Tabuk",
    "Hail",
    "Northern Borders",
    "Jazan",
    "Najran",
    "Al-Bahah",
    "Al-Jouf",
]

REGION_MAP = {
    "Riyadh": "Riyadh",
    "Makkah": "Makkah",
    "Madinah": "Madinah",
    "Qassim": "Al-Qassim",
    "Al Qassim": "Al-Qassim",
    "Al-Qassim": "Al-Qassim",
    "Eastern Region": "Eastern Region",
    "Eastern": "Eastern Region",
    "Aseer": "Asir",
    "Asir": "Asir",
    "Tabuk": "Tabuk",
    "Hail": "Hail",
    "Northern Border": "Northern Borders",
    "Northern Borders": "Northern Borders",
    "Jazan": "Jazan",
    "Najran": "Najran",
    "Al-Baha": "Al-Bahah",
    "Al Bahah": "Al-Bahah",
    "Al-Bahah": "Al-Bahah",
    "Al Jouf": "Al-Jouf",
    "Al-Jawf": "Al-Jouf",
    "Al-Jouf": "Al-Jouf",
}


def normalize_region(value):
    """Return a canonical administrative-region label without guessing unknown names."""
    if pd.isna(value):
        return pd.NA
    cleaned = re.sub(r"\s+", " ", str(value).strip().replace("–", "-").replace("—", "-"))
    return REGION_MAP.get(cleaned, cleaned)


def read_csv_flexible(path):
    """Read a supplied CSV while detecting comma/semicolon delimiters."""
    for encoding in ("utf-8-sig", "utf-8", "cp1256", "latin-1"):
        try:
            return pd.read_csv(path, sep=None, engine="python", encoding=encoding)
        except UnicodeDecodeError:
            continue
    raise UnicodeError(f"Unable to decode {path}")


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def extract_gastat_household(year):
    """Extract one original GASTAT regional household table into canonical long form."""
    setting = GASTAT_TABLES[year]
    path = GASTAT_DIR / setting["file"]
    if not path.is_file():
        raise FileNotFoundError(path)
    workbook = openpyxl.load_workbook(path, read_only=True, data_only=True)
    if setting["sheet"] not in workbook.sheetnames:
        raise KeyError(f"Missing sheet {setting['sheet']} in {path.name}")
    sheet = workbook[setting["sheet"]]
    start_row, end_row = setting["data_rows"]
    records = []
    for excel_row, row in enumerate(
        sheet.iter_rows(min_row=start_row, max_row=end_row, values_only=True),
        start=start_row,
    ):
        values = list(row)
        if len(values) < 6 or values[1] is None:
            continue
        original_region = str(values[1]).strip()
        region = normalize_region(original_region)
        measures = [
            ("Winter", "Consumption", "kWh", values[2]),
            ("Winter", "Cost", "SAR", values[3]),
            ("Rest of the year", "Consumption", "kWh", values[4]),
            ("Rest of the year", "Cost", "SAR", values[5]),
        ]
        for season, measure, unit, raw_value in measures:
            records.append(
                {
                    "year": year,
                    "region_original": original_region,
                    "region": region,
                    "season": season,
                    "measure": measure,
                    "unit": unit,
                    "value": pd.to_numeric(raw_value, errors="coerce"),
                    "source_type": "GASTAT original workbook",
                    "source_file": setting["file"],
                    "source_sheet": setting["sheet"],
                    "source_table": setting["table"],
                    "source_excel_row": excel_row,
                }
            )
    result = pd.DataFrame(records)
    expected = len(CANONICAL_REGIONS) * 2 * 2
    if len(result) != expected:
        raise ValueError(f"{year} GASTAT extraction produced {len(result)} rows; expected {expected}")
    if set(result["region"]) != set(CANONICAL_REGIONS):
        raise ValueError(f"{year} GASTAT region coverage is not canonical")
    return result.sort_values(["year", "region", "season", "measure"]).reset_index(drop=True)


def read_household_csv():
    """Read the supplied 2017–2022 long household CSV independently."""
    if not HOUSEHOLD_CSV_PATH.is_file():
        raise FileNotFoundError(HOUSEHOLD_CSV_PATH)
    frame = read_csv_flexible(HOUSEHOLD_CSV_PATH)
    expected_columns = {
        "Year", "Administrative Region", "Season", "Measure and Unit", "Value"
    }
    if not expected_columns.issubset(frame.columns):
        raise ValueError(
            f"Household CSV columns changed. Found: {frame.columns.tolist()}"
        )
    frame = frame.rename(
        columns={
            "Year": "year",
            "Administrative Region": "region_original",
            "Season": "season",
            "Value": "value",
        }
    )
    frame["year"] = pd.to_numeric(frame["year"], errors="coerce").astype("Int64")
    frame["region"] = frame["region_original"].map(normalize_region)
    parsed = frame["Measure and Unit"].str.extract(
        r"(?P<measure>Consumption|Cost)\s*\((?P<unit>[^)]+)\)",
        expand=True,
    )
    frame["measure"] = parsed["measure"]
    frame["unit"] = parsed["unit"].replace({"KWh": "kWh", "kWh": "kWh"})
    frame["value"] = pd.to_numeric(frame["value"], errors="coerce")
    frame["source_type"] = "Independent supplied household CSV"
    frame["source_file"] = HOUSEHOLD_CSV_NAME
    frame["source_sheet"] = pd.NA
    frame["source_table"] = pd.NA
    frame["source_excel_row"] = pd.NA
    ordered = [
        "year", "region_original", "region", "season", "measure", "unit", "value",
        "source_type", "source_file", "source_sheet", "source_table", "source_excel_row",
    ]
    return frame[ordered].sort_values(
        ["year", "region", "season", "measure"]
    ).reset_index(drop=True)


print(f"Project root: {PROJECT_ROOT}")
print(f"Run started: {RUN_STARTED.isoformat(timespec='seconds')}")


In [ ]:
from PIL import Image, ImageDraw, ImageFont


def _font(size=18, bold=False):
    candidates = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold
        else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/System/Library/Fonts/Supplemental/Arial Bold.ttf" if bold
        else "/System/Library/Fonts/Supplemental/Arial.ttf",
    ]
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return ImageFont.truetype(str(path), size=size)
    return ImageFont.load_default()


def save_bar_chart(labels, values, title, x_label, output_path, color="#3267A8"):
    """Create a dependency-light publication-ready horizontal bar chart."""
    width = 1500
    row_height = 52
    height = max(650, 180 + len(labels) * row_height)
    image = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(image)
    title_font = _font(30, bold=True)
    body_font = _font(18)
    small_font = _font(16)
    draw.text((60, 35), title, fill="#152238", font=title_font)
    left, right, top = 430, width - 100, 120
    max_value = max([float(v) for v in values] + [1.0])
    for index, (label, value) in enumerate(zip(labels, values)):
        y = top + index * row_height
        draw.text((50, y + 8), str(label)[:42], fill="#1F2937", font=small_font)
        bar_width = int((right - left) * float(value) / max_value)
        draw.rounded_rectangle(
            (left, y + 6, left + max(bar_width, 2), y + 36),
            radius=5,
            fill=color,
        )
        draw.text(
            (left + bar_width + 12, y + 8),
            f"{float(value):,.0f}",
            fill="#111827",
            font=small_font,
        )
    draw.text((left, height - 55), x_label, fill="#374151", font=body_font)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    image.save(output_path, format="PNG", optimize=True)


def save_matrix_chart(matrix, row_labels, column_labels, title, output_path):
    """Create a labeled blue heatmap for categorical coverage."""
    cell_w, cell_h = 95, 38
    left, top = 260, 140
    width = left + len(column_labels) * cell_w + 100
    height = top + len(row_labels) * cell_h + 90
    image = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(image)
    draw.text((45, 35), title, fill="#152238", font=_font(28, bold=True))
    for col, label in enumerate(column_labels):
        draw.text((left + col * cell_w + 8, 95), str(label), fill="#1F2937", font=_font(15, True))
    max_value = max(float(np.nanmax(matrix)), 1.0)
    for row, label in enumerate(row_labels):
        y = top + row * cell_h
        draw.text((35, y + 8), str(label), fill="#1F2937", font=_font(14))
        for col in range(len(column_labels)):
            value = float(matrix[row][col])
            intensity = int(235 - 165 * value / max_value)
            fill = (intensity, min(245, intensity + 12), 245)
            x = left + col * cell_w
            draw.rectangle((x, y, x + cell_w - 3, y + cell_h - 3), fill=fill)
            draw.text((x + 32, y + 8), f"{value:.0f}", fill="#111827", font=_font(13))
    output_path.parent.mkdir(parents=True, exist_ok=True)
    image.save(output_path, format="PNG", optimize=True)


## 2. Audit CSV and Excel Sources


In [ ]:
audit_rows = []
year_coverage_rows = []

for path in sorted(RAW_DIR.rglob("*")):
    if not path.is_file() or path.name == ".DS_Store" or path.name.startswith("._"):
        continue
    relative = path.relative_to(PROJECT_ROOT).as_posix()
    authoritative = "technical_work_reference" not in relative
    if path.suffix.lower() == ".csv":
        frame = read_csv_flexible(path)
        metadata_rows = 0
        footer_rows = 0
        for _, row in frame.iterrows():
            text = " ".join(str(value) for value in row.tolist() if pd.notna(value)).lower()
            if any(token in text for token in ("data set name", "أسم مجموعة البيانات", "من (السنه)", "إلى (السنة)")):
                metadata_rows += 1
            if any(token in text for token in ("source:", "reference:", "return to index", "back to index")):
                footer_rows += 1
        years = []
        for candidate in ("Year", "year", "السنة", "Date"):
            if candidate in frame.columns:
                if candidate == "Date":
                    years = pd.to_datetime(frame[candidate], errors="coerce").dt.year.dropna().astype(int).tolist()
                else:
                    years = pd.to_numeric(frame[candidate], errors="coerce").dropna().astype(int).tolist()
                break
        audit_rows.append(
            {
                "file_name": path.name,
                "file_type": "CSV",
                "authoritative": authoritative,
                "rows_or_sheets": len(frame),
                "columns": frame.shape[1],
                "missing_cells": int(frame.isna().sum().sum()),
                "duplicate_rows": int(frame.duplicated().sum()),
                "metadata_rows_detected": metadata_rows,
                "footer_rows_detected": footer_rows,
                "formatting_issue": "Delimiter/encoding and numeric-text parsing require explicit handling",
            }
        )
        for year in sorted(set(years)):
            year_coverage_rows.append({"file_name": path.name, "year": year})
    elif path.suffix.lower() == ".xlsx":
        workbook = openpyxl.load_workbook(path, read_only=True, data_only=True)
        inflated_sheets = sum(
            1 for sheet_name in workbook.sheetnames
            if workbook[sheet_name].max_row >= 1000
        )
        audit_rows.append(
            {
                "file_name": path.name,
                "file_type": "XLSX",
                "authoritative": authoritative,
                "rows_or_sheets": len(workbook.sheetnames),
                "columns": pd.NA,
                "missing_cells": pd.NA,
                "duplicate_rows": pd.NA,
                "metadata_rows_detected": pd.NA,
                "footer_rows_detected": "Present in statistical tables",
                "formatting_issue": (
                    f"{inflated_sheets} sheets have formatting extended to >=1000 rows; "
                    "tables also use multirow headers, merged cells, titles, footnotes, and totals"
                ),
            }
        )
        match = re.search(r"(20\d{2})", path.name)
        if match:
            year_coverage_rows.append({"file_name": path.name, "year": int(match.group(1))})

audit_summary = pd.DataFrame(audit_rows)
year_coverage = pd.DataFrame(year_coverage_rows).drop_duplicates()
display(audit_summary)


## 3. Region and Seasonal Coverage


In [ ]:
household_csv = read_household_csv()
household_data = household_csv[
    household_csv["region_original"].astype(str).str.lower().ne("total")
].copy()

unknown_regions = sorted(set(household_data["region"]) - set(CANONICAL_REGIONS))
if unknown_regions:
    raise ValueError(f"Unresolved household CSV region labels: {unknown_regions}")

region_mapping_audit = (
    household_data[["region_original", "region"]]
    .drop_duplicates()
    .sort_values(["region", "region_original"])
)
season_coverage = (
    household_data.groupby(["year", "region"])["season"]
    .nunique()
    .rename("season_count")
    .reset_index()
)
measure_coverage = (
    household_data.groupby(["year", "region", "season"])["measure"]
    .nunique()
    .rename("measure_count")
    .reset_index()
)

print("Region mappings:")
display(region_mapping_audit)
print("Season coverage distribution:", season_coverage["season_count"].value_counts().to_dict())
print("Measure coverage distribution:", measure_coverage["measure_count"].value_counts().to_dict())


## 4. GASTAT/CSV Overlap and Conflict Detection


In [ ]:
gastat = pd.concat(
    [extract_gastat_household(year) for year in sorted(GASTAT_TABLES)],
    ignore_index=True,
)
csv_overlap = household_data[household_data["year"].isin(GASTAT_TABLES)].copy()
keys = ["year", "region", "season", "measure", "unit"]

conflict_report = gastat[keys + ["value", "source_file", "source_sheet", "source_excel_row"]].merge(
    csv_overlap[keys + ["value", "source_file"]],
    on=keys,
    how="outer",
    suffixes=("_gastat", "_csv"),
    indicator=True,
    validate="one_to_one",
)
conflict_report["absolute_difference"] = (
    conflict_report["value_gastat"] - conflict_report["value_csv"]
).abs()
conflict_report["percentage_difference"] = np.where(
    conflict_report["value_gastat"].abs() > 0,
    conflict_report["absolute_difference"] / conflict_report["value_gastat"].abs() * 100,
    np.nan,
)
ABSOLUTE_TOLERANCE = 1e-3
conflict_report["conflict_status"] = np.select(
    [
        conflict_report["_merge"].ne("both"),
        conflict_report["absolute_difference"].gt(ABSOLUTE_TOLERANCE),
    ],
    ["Missing from one source", "Conflict above tolerance"],
    default="Agreement within tolerance",
)
conflict_report["requires_decision"] = conflict_report["conflict_status"].ne(
    "Agreement within tolerance"
)

conflict_path = RESULTS_DIR / "conflict_report.csv"
conflict_report.to_csv(conflict_path, index=False)
display(conflict_report["conflict_status"].value_counts().to_frame("observations"))
print(f"Maximum absolute difference: {conflict_report['absolute_difference'].max():.9f}")


## 5. Generate Quality Visualizations


In [ ]:
plot_audit = audit_summary.copy()
plot_audit["missing_cells_numeric"] = pd.to_numeric(
    plot_audit["missing_cells"], errors="coerce"
).fillna(0)
save_bar_chart(
    plot_audit["file_name"].tolist(),
    plot_audit["missing_cells_numeric"].tolist(),
    "Missing Cells Detected in Raw Tabular Sources",
    "Missing cell count (Excel workbook-wide counts are not inferred from formatting)",
    FIGURES_DIR / "missing_values.png",
    color="#B24C63",
)

mapping_labels = [
    f"{row.region_original} → {row.region}"
    for row in region_mapping_audit.itertuples(index=False)
]
save_bar_chart(
    mapping_labels,
    [1] * len(mapping_labels),
    "Administrative Region Name Standardization",
    "One documented mapping per supplied raw label",
    FIGURES_DIR / "region_name_mapping.png",
    color="#2D8C75",
)

coverage_table = (
    year_coverage.assign(present=1)
    .pivot_table(index="file_name", columns="year", values="present", aggfunc="max", fill_value=0)
    .sort_index()
)
save_matrix_chart(
    coverage_table.to_numpy(),
    coverage_table.index.tolist(),
    coverage_table.columns.tolist(),
    "Temporal Coverage by Raw Dataset",
    FIGURES_DIR / "year_coverage.png",
)


## 6. Write the Quality Report


In [ ]:
conflicts = int(conflict_report["requires_decision"].sum())
report_lines = [
    "# Phase 1 Data Quality Report",
    "",
    f"Generated: {datetime.now().astimezone().isoformat(timespec='seconds')}",
    "",
    "## Scope",
    "",
    "All files under `data/raw/` were inventoried. Historical files under "
    "`technical_work_reference/` were audited as non-authoritative artifacts and were not used "
    "as primary evidence.",
    "",
    "## GASTAT and household CSV overlap",
    "",
    f"- Overlapping observations compared: {len(conflict_report):,}",
    f"- Observations requiring a scientific source decision: {conflicts:,}",
    f"- Maximum absolute numeric difference: {conflict_report['absolute_difference'].max():.9f}",
    "- Tolerance: 0.001 in the reported unit.",
    "",
    "The supplied CSV agrees with the original GASTAT 2019–2022 regional seasonal tables "
    "within numeric tolerance. Notebook 02 therefore selects GASTAT for overlapping years "
    "because it is the original publication, while retaining the CSV value and comparison evidence.",
    "",
    "## Household panel coverage",
    "",
    f"- Raw household CSV years: {int(household_data['year'].min())}–{int(household_data['year'].max())}",
    f"- Canonical administrative regions: {household_data['region'].nunique()}",
    f"- Region-year combinations: {household_data[['year','region']].drop_duplicates().shape[0]}",
    f"- Season counts other than two: {(season_coverage['season_count'] != 2).sum()}",
    f"- Measure counts other than two: {(measure_coverage['measure_count'] != 2).sum()}",
    "",
    "## Important quality findings",
    "",
    "- GASTAT workbooks use presentation layouts with multirow headers, titles, totals, footers, "
    "merged cells, and in 2020–2021 large formatted empty regions.",
    "- Region spelling varies across years and sources; all mappings are explicit and no fuzzy "
    "matching is used.",
    "- The national electricity-user CSV contains blank/metadata rows and a scale discontinuity "
    "in 2023–2024 that must be resolved before analytical use.",
    "- The 2023 four-operating-region consumption file has an unclear unit definition and remains "
    "independent.",
    "- Administrative-region household data must not be merged with four-region electricity-system "
    "data without an authoritative crosswalk.",
    "",
    "## Formatting and metadata",
    "",
    "Metadata rows and source/footer text are documented rather than silently treated as observations. "
    "Original raw files remain unchanged.",
]

quality_path = RESULTS_DIR / "quality_report.md"
quality_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

required = [
    quality_path,
    conflict_path,
    FIGURES_DIR / "missing_values.png",
    FIGURES_DIR / "region_name_mapping.png",
    FIGURES_DIR / "year_coverage.png",
]
for path in required:
    if not path.is_file() or path.stat().st_size == 0:
        raise IOError(f"Required output missing or empty: {path}")

print("Notebook 01 complete")
for path in required:
    print("-", path.relative_to(PROJECT_ROOT))
